In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/visual_rag

In [ ]:
# install all packages
%pip install colpali-engine transformers peft accelerate pymupdf pillow psycopg2-binary pgvector

In [ ]:
# install postgres
%%bash
set -e
apt-get update -qq
apt-get install -y postgresql postgresql-server-dev-all build-essential git

service postgresql start

if [ ! -d /content/pgvector ]; then
    git clone --branch v0.8.1 https://github.com/pgvector/pgvector.git /content/pgvector
fi

cd /content/pgvector
make
make install

sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'colab_local';"

In [ ]:
%pip uninstall -y torchao

In [ ]:
# to load data
!python data.py

In [ ]:
# save database
!PGPASSWORD=colab_local pg_dump -h localhost -U postgres -d postgres -Fc -f database_backup.dump

In [ ]:
# to reload database
!PGPASSWORD=colab_local pg_restore -h localhost -U postgres -d postgres --no-owner database_backup.dump

In [ ]:
# test database
import database

conn = database.create_connection()

try:
    with conn.cursor() as curr:
        curr.execute("""
            SELECT
                d.id,
                d.name,
                COUNT(DISTINCT p.id) AS pages,
                COUNT(e.page_id) AS embedding_vectors
            FROM documents d
            LEFT JOIN pages p ON p.docid = d.id
            LEFT JOIN page_embeddings e ON e.page_id = p.id
            GROUP BY d.id, d.name
            ORDER BY d.id
        """)

        for row in curr.fetchall():
            print(row)
finally:
    conn.close()

In [ ]:
# to chat
!python chat.py

In [ ]:
# to run all 3 baselines
!python run_baseline.py